In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.naive_bayes import GaussianNB, MultinomialNB, ComplementNB
from sklearn.metrics import f1_score
import warnings

warnings.filterwarnings('ignore')
COLUNA_ALVO = 'label'

bases_dados = [
    'HOG_128_16x16.csv',
    'HOG_128_32x32.csv',
    'HOG_256_16x16.csv',
    'HOG_256_32x32.csv',
    'PCA_075_HOG_128_16x16.csv',
    'PCA_075_HOG_128_32x32.csv',
    'PCA_075_HOG_256_16x16.csv',
    'PCA_075_HOG_256_32x32.csv',
    'PCA_090_HOG_128_16x16.csv',
    'PCA_090_HOG_128_32x32.csv',
    'PCA_090_HOG_256_16x16.csv',
    'PCA_090_HOG_256_32x32.csv'
]


classificadores = {
    'GaussianNB': GaussianNB(),
    'MultinomialNB': MultinomialNB(),
    'ComplementNB': ComplementNB()
}

resultados_finais = []
for base in bases_dados:
    if not os.path.exists(base):
        print(f"Arquivo não encontrado: {base}.")
        continue

    print(f"Processando: {base}...")
    df = pd.read_csv(base, on_bad_lines='skip')

    df = df.dropna(subset=[COLUNA_ALVO])

    X = df.drop(columns=[COLUNA_ALVO])

    y = df[COLUNA_ALVO].astype(str)

    X = X.apply(pd.to_numeric, errors='coerce').fillna(0)

    for nome_clf, clf in classificadores.items():
        try:
            scores_cv = cross_val_score(clf, X, y, cv=10, scoring='f1_macro')
            f1_10fold = np.mean(scores_cv)
        except ValueError as e:
            f1_10fold = "Erro"

        try:
            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)
            clf.fit(X_train, y_train)
            y_pred = clf.predict(X_test)
            f1_holdout = f1_score(y_test, y_pred, average='macro')
        except ValueError:
            f1_holdout = "Erro"

        resultados_finais.append({
            'Base': base,
            'Classificador': nome_clf,
            '10-fold CV': f1_10fold,
            '70/30': f1_holdout
        })

df_resultados = pd.DataFrame(resultados_finais)

print("\n" + "="*50)
print("RESULTADOS FINAIS PRONTOS PARA A TABELA")
print("="*50)
display(df_resultados)

df_resultados.to_csv('resultados_finais_para_tabela.csv', index=False)
print("\nArquivo 'resultados_finais_para_tabela.csv' gerado")

Processando: HOG_128_16x16.csv...
Processando: HOG_128_32x32.csv...
Processando: HOG_256_16x16.csv...
Processando: HOG_256_32x32.csv...
Processando: PCA_075_HOG_128_16x16.csv...
Processando: PCA_075_HOG_128_32x32.csv...
Processando: PCA_075_HOG_256_16x16.csv...
Processando: PCA_075_HOG_256_32x32.csv...
Processando: PCA_090_HOG_128_16x16.csv...
Processando: PCA_090_HOG_128_32x32.csv...
Processando: PCA_090_HOG_256_16x16.csv...
Processando: PCA_090_HOG_256_32x32.csv...

RESULTADOS FINAIS PRONTOS PARA A TABELA


,Base,Classificador,10-fold CV,70/30
0,HOG_128_16x16.csv,GaussianNB,0.725739,0.724694
1,HOG_128_16x16.csv,MultinomialNB,0.710434,0.72079
2,HOG_128_16x16.csv,ComplementNB,0.710434,0.72079
3,HOG_128_32x32.csv,GaussianNB,0.704401,0.729124
4,HOG_128_32x32.csv,MultinomialNB,0.670211,0.724061
5,HOG_128_32x32.csv,ComplementNB,0.670211,0.712495
6,HOG_256_16x16.csv,GaussianNB,0.723759,0.724981
7,HOG_256_16x16.csv,MultinomialNB,0.711541,0.708313
8,HOG_256_16x16.csv,ComplementNB,0.711541,0.708313
9,HOG_256_32x32.csv,GaussianNB,0.769795,0.794075



Arquivo 'resultados_finais_para_tabela.csv' gerado com sucesso!


In [ ]:
import pandas as pd
import numpy as np

try:
    df_resultados = pd.read_csv('resultados_finais_para_tabela.csv')
except FileNotFoundError:
    print("Arquivo 'resultados_finais_para_tabela.csv' não encontrado. Rode o código principal primeiro")

df_resultados['10-fold CV'] = pd.to_numeric(df_resultados['10-fold CV'], errors='coerce')
df_resultados['70/30'] = pd.to_numeric(df_resultados['70/30'], errors='coerce')


df_melt = df_resultados.melt(id_vars=['Base', 'Classificador'],
                             value_vars=['10-fold CV', '70/30'],
                             var_name='Treinamento/Teste',
                             value_name='F1_Score')

tabela_figura1 = df_melt.pivot_table(index=['Base', 'Treinamento/Teste'],
                                     columns='Classificador',
                                     values='F1_Score',
                                     aggfunc='first')

medias = tabela_figura1.mean()
desvios = tabela_figura1.std()

df_estatisticas = pd.DataFrame({
    'Média =>': medias,
    'Desv. Pad. =>': desvios
}).T # .T inverte linhas e colunas para ficar igual ao rodapé do PDF

print("\n" + "="*60)
print("ESTATÍSTICAS FINAIS PARA O RODAPÉ DA TABELA")
print("="*60)
display(df_estatisticas)

TABELA DE RESULTADOS (FORMATO DA FIGURA 1)


Classificador                                ComplementNB  GaussianNB  \
Base                      Treinamento/Teste                             
HOG_128_16x16.csv         10-fold CV             0.710434    0.725739   
                          70/30                  0.720790    0.724694   
HOG_128_32x32.csv         10-fold CV             0.670211    0.704401   
                          70/30                  0.712495    0.729124   
HOG_256_16x16.csv         10-fold CV             0.711541    0.723759   
                          70/30                  0.708313    0.724981   
HOG_256_32x32.csv         10-fold CV             0.727285    0.769795   
                          70/30                  0.741608    0.794075   
PCA_075_HOG_128_16x16.csv 10-fold CV                  NaN    0.669177   
                          70/30                       NaN    0.665529   
PCA_075_HOG_128_32x32.csv 10-fold CV                  NaN    0.698811   
                          70/30                       NaN    0.720828   
PCA_075_HOG_256_16x16.csv 10-fold CV                  NaN    0.627916   
                          70/30                       NaN    0.627297   
PCA_075_HOG_256_32x32.csv 10-fold CV                  NaN    0.685574   
                          70/30                       NaN    0.670553   
PCA_090_HOG_128_16x16.csv 10-fold CV                  NaN    0.635590   
                          70/30                       NaN    0.610546   
PCA_090_HOG_128_32x32.csv 10-fold CV                  NaN    0.673207   
                          70/30                       NaN    0.716174   
PCA_090_HOG_256_16x16.csv 10-fold CV                  NaN    0.611697   
                          70/30                       NaN    0.610546   
PCA_090_HOG_256_32x32.csv 10-fold CV                  NaN    0.655735   
                          70/30                       NaN    0.666644   

Classificador                                MultinomialNB  
Base                      Treinamento/Teste                 
HOG_128_16x16.csv         10-fold CV              0.710434  
                          70/30                   0.720790  
HOG_128_32x32.csv         10-fold CV              0.670211  
                          70/30                   0.724061  
HOG_256_16x16.csv         10-fold CV              0.711541  
                          70/30                   0.708313  
HOG_256_32x32.csv         10-fold CV              0.738280  
                          70/30                   0.737380  
PCA_075_HOG_128_16x16.csv 10-fold CV                   NaN  
                          70/30                        NaN  
PCA_075_HOG_128_32x32.csv 10-fold CV                   NaN  
                          70/30                        NaN  
PCA_075_HOG_256_16x16.csv 10-fold CV                   NaN  
                          70/30                        NaN  
PCA_075_HOG_256_32x32.csv 10-fold CV                   NaN  
                          70/30                        NaN  
PCA_090_HOG_128_16x16.csv 10-fold CV                   NaN  
                          70/30                        NaN  
PCA_090_HOG_128_32x32.csv 10-fold CV                   NaN  
                          70/30                        NaN  
PCA_090_HOG_256_16x16.csv 10-fold CV                   NaN  
                          70/30                        NaN  
PCA_090_HOG_256_32x32.csv 10-fold CV                   NaN  
                          70/30                        NaN


ESTATÍSTICAS FINAIS PARA O RODAPÉ DA TABELA


Classificador,ComplementNB,GaussianNB,MultinomialNB
Média =>,0.712834,0.685100,0.715126
Desv. Pad. =>,0.020511,0.050154,0.021520
